# Article-Update — 3 OOS Periods (COVID Extended to 2019–2020)

**Pipeline analysis** for the SBTS deep hedging article. Replaces the previous 4-period scheme.

**3 OOS periods:**

| Period | Range | Notes |
|---|---|---|
| `COVID_2019_2020` | 2019-01-01 → 2020-12-31 | Extended to 2 years for full crash coverage |
| `PostCOVID_2021_22` | 2021-01-01 → 2022-12-31 | Unchanged |
| `Recent_2023_25` | 2023-01-01 → 2025-12-31 | Unchanged |

**Key changes vs `article_3periods_fresh`:**

1. **COVID range extended** from 1 year (2020) to 2 years (2019-2020) — the original 1-year setup had only ~14% of windows containing the full Feb-Apr 2020 crash episode; the rest were post-crash recovery dynamics that diluted σ_COVID. Extending to 2019-2020 includes both pre-crash 2019 calm and the crash itself, increasing windows containing crash content.
2. **All 360 checkpoints re-evaluated on all 3 periods** — no reuse of training-time JSONs. Guarantees consistency since COVID range changed.
3. **Seed=3 outlier swap baked into eval cell** — SBTS/asian_worst_of_put/κ=0.95/seed=3 diverged in CVaR phase; cell loads seed=10's converged checkpoint but stores under seed=3 key (paired t-tests preserve n=10).
4. **Stress test removed** — no SDR/TRAF/CPSR/CRG metrics, no `tab_stress_*` tables, no `fig4_stress_degradation`. Only statistical tests (paired t-test + BH-FDR) and MCS retained.

**Output structure (`article_results_3p_covid_ext/`):**

- `per_seed_metrics_3p_covid_ext.csv` — master DataFrame
- `statistical_tests_3p_covid_ext.csv` — 216 paired t-tests with BH-FDR
- `mcs_3p_covid_ext.csv` — Hansen-Lunde-Nason MCS for 18 cells
- `tab_main_results.tex`, `tab_scoreboard.tex`, `tab_scoreboard_detail.tex`, `tab_mcs.tex`, `tab_v0.tex`, `tab_curriculum.tex`
- `figures/fig_performance_across_regimes.{pdf,png}`

**Estimated runtime on Colab T4:** ~5 minutes (most of it is CELL 4 re-evaluation: 1,080 hedge passes through the network).

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup, paths, constants
#
# 3 OOS periods (UPDATED — COVID extended to full 2 years):
#   COVID_2019_2020   = 2019-01-01 → 2020-12-31
#   PostCOVID_2021_22 = 2021-01-01 → 2022-12-31
#   Recent_2023_25    = 2023-01-01 → 2025-12-31
# ═══════════════════════════════════════════════════════════════════
import os, json, hashlib, warnings, time, datetime
from pathlib import Path
from itertools import product
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

# ── Paths (Colab + local fallback) ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_FOLDER = Path('/content/drive/MyDrive/ARTICLE_SBTS')
    IN_COLAB = True
except ImportError:
    DRIVE_FOLDER = Path('./ARTICLE_SBTS')
    IN_COLAB = False

# Run tag — distinguishes outputs from previous 3p run
RUN_TAG = '3p_covid_ext'

CKPT_ROOT     = DRIVE_FOLDER / 'checkpoints_article'
RESULTS_DIR   = DRIVE_FOLDER / f'article_results_{RUN_TAG}'
FIG_DIR       = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Experimental grid ─────────────────────────────────────────────
DS_NAMES     = ['GBM', 'Heston', 'SBTS']
OPTION_NAMES = ['basket_asian_call', 'asian_worst_of_put']
KAPPA_LEVELS = [0.95, 1.00, 1.05]
N_SEEDS      = 10
PHASES       = ['mse', 'cvar']

# ── 3 OOS periods (UPDATED — extended COVID range) ───────────────
PERIODS = ['COVID_2019_2020', 'PostCOVID_2021_22', 'Recent_2023_25']

PERIOD_DATE_RANGES = {
    'COVID_2019_2020':   ('2019-01-01', '2020-12-31'),
    'PostCOVID_2021_22': ('2021-01-01', '2022-12-31'),
    'Recent_2023_25':    ('2023-01-01', '2025-12-31'),
}

PERIOD_LABELS = {
    'COVID_2019_2020':   'COVID 2019--2020',
    'PostCOVID_2021_22': 'PostCOVID 2021--2022',
    'Recent_2023_25':    'Recent 2023--2025',
}

PERIOD_SHORT = {
    'COVID_2019_2020':   'COVID',
    'PostCOVID_2021_22': 'PostCOVID',
    'Recent_2023_25':    'Recent',
}

OPTION_LABELS = {
    'basket_asian_call':  'Basket call',
    'asian_worst_of_put': 'Worst-of put',
}

# Reference period for tables that need a single period (V0, curriculum)
BASELINE_PERIOD = 'Recent_2023_25'

# ── Hyperparameters (must match training) ─────────────────────────
T          = 252
N_ASSETS   = 3
COST_RATE  = 0.001
TICKERS    = ['AAPL', 'JPM', 'XOM']

# ── Plot style ────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 10, 'font.family': 'serif',
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.5,
})
DS_COLORS = {'GBM': '#E74C3C', 'Heston': '#27AE60', 'SBTS': '#2980B9'}

print(f"Drive folder : {DRIVE_FOLDER}")
print(f"Checkpoints  : {CKPT_ROOT}")
print(f"Outputs      : {RESULTS_DIR}")
print(f"Run tag      : {RUN_TAG}")
print(f"\nPeriods (all will be re-evaluated on every checkpoint):")
for p, (s, e) in PERIOD_DATE_RANGES.items():
    print(f"  {p:<22s} {s} → {e}")


Mounted at /content/drive
Drive folder : /content/drive/MyDrive/ARTICLE_SBTS
Checkpoints  : /content/drive/MyDrive/ARTICLE_SBTS/checkpoints_article
Outputs      : /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext
Run tag      : 3p_covid_ext

Periods (all will be re-evaluated on every checkpoint):
  COVID_2019_2020        2019-01-01 → 2020-12-31
  PostCOVID_2021_22      2021-01-01 → 2022-12-31
  Recent_2023_25         2023-01-01 → 2025-12-31


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Build historical paths for all 3 periods
#
# COVID range mới (2019-01 → 2020-12) → cache mới với RUN_TAG.
# PostCOVID/Recent có cùng range với phiên bản 3p trước → reuse legacy
# cache nếu có.
# ═══════════════════════════════════════════════════════════════════
HIST_PATHS_FILE   = DRIVE_FOLDER / f'historical_test_paths_{RUN_TAG}.npz'
HIST_PATHS_LEGACY = DRIVE_FOLDER / 'historical_test_paths_3p.npz'

NEEDED_KEYS = [f'S_norm_{p}' for p in PERIODS]


def build_sliding_paths(start_date, end_date, tickers=TICKERS, T_days=T):
    """Sliding T_days windows starting on every trading day in
    [start_date, end_date]. Downloads `T_days * 1.6` extra calendar days
    beyond end_date so each in-period start has full lookahead. Each
    window is normalised to S_0 = 1."""
    import yfinance as yf
    end_buffer = (pd.Timestamp(end_date)
                  + pd.DateOffset(days=int(T_days * 1.6))).strftime('%Y-%m-%d')
    print(f"  Downloading {tickers} {start_date} → {end_buffer} "
          f"(incl. ~{int(T_days*1.6)}d lookahead) ...")
    data = yf.download(tickers, start=start_date, end=end_buffer,
                       auto_adjust=True, progress=False)['Close']
    data = data[tickers]
    prices = data.values
    dates  = data.index
    n_total, d = prices.shape

    end_ts = pd.Timestamp(end_date)
    in_period_mask = dates <= end_ts
    n_starts = int(in_period_mask.sum())

    paths_list = []
    for i in range(n_starts):
        if i + T_days + 1 > n_total:
            break
        paths_list.append(prices[i : i + T_days + 1] / prices[i])
    paths = np.asarray(paths_list, dtype=np.float32)

    n_w = len(paths)
    print(f"    starts in period: {n_starts}, windows built: {n_w}, "
          f"shape: {paths.shape}")
    if n_w == 0:
        raise ValueError("No windows built — extend lookahead or shorten end_date.")
    return paths


# ── Resolve cache ────────────────────────────────────────────────
out = {}

if HIST_PATHS_FILE.exists():
    print(f"✓ Cache hit: {HIST_PATHS_FILE.name}")
    cached = np.load(HIST_PATHS_FILE)
    out = {k: cached[k] for k in NEEDED_KEYS if k in cached.files}
    missing = [k for k in NEEDED_KEYS if k not in cached.files]
elif HIST_PATHS_LEGACY.exists():
    print(f"Legacy cache found: {HIST_PATHS_LEGACY.name}")
    legacy = np.load(HIST_PATHS_LEGACY)
    print(f"  Legacy keys: {sorted(legacy.files)}")
    # Reuse only PostCOVID + Recent (same ranges); COVID range changed → must rebuild
    reuse_keys = ['S_norm_PostCOVID_2021_22', 'S_norm_Recent_2023_25']
    out = {k: legacy[k] for k in reuse_keys if k in legacy.files}
    missing = [k for k in NEEDED_KEYS if k not in out]
    if missing:
        print(f"  → migrating {len(out)} keys from legacy, "
              f"will build {len(missing)} (incl. extended COVID)")
else:
    print("No cache → building all 3 periods from yfinance")
    missing = list(NEEDED_KEYS)

for k in missing:
    period = k.replace('S_norm_', '')
    sd, ed = PERIOD_DATE_RANGES[period]
    print(f"\nBuilding {period} ({sd} → {ed}) ...")
    out[k] = build_sliding_paths(sd, ed)

np.savez_compressed(HIST_PATHS_FILE, **out)
print(f"\n✓ Saved: {HIST_PATHS_FILE}")
for k in NEEDED_KEYS:
    print(f"  {k}: {out[k].shape}")

for k in NEEDED_KEYS:
    if out[k].shape[0] < 10:
        print(f"  ⚠ {k} has only {out[k].shape[0]} paths")


Legacy cache found: historical_test_paths_3p.npz
  Legacy keys: ['S_norm_COVID_2020', 'S_norm_PostCOVID_2021_22', 'S_norm_Recent_2023_25']
  → migrating 2 keys from legacy, will build 1 (incl. extended COVID)

Building COVID_2019_2020 (2019-01-01 → 2020-12-31) ...
    starts in period: 505, windows built: 505, shape: (505, 253, 3)

✓ Saved: /content/drive/MyDrive/ARTICLE_SBTS/historical_test_paths_3p_covid_ext.npz
  S_norm_COVID_2019_2020: (505, 253, 3)
  S_norm_PostCOVID_2021_22: (503, 253, 3)
  S_norm_Recent_2023_25: (499, 253, 3)


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Model components (HedgingNetwork + payoffs + evaluator)
# Copied verbatim from training pipeline so metrics match by construction.
# ═══════════════════════════════════════════════════════════════════
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


def compute_running_averages(S_paths):
    """Causal running averages (no look-ahead)."""
    M, T1, d = S_paths.shape
    T_ = T1 - 1
    avg = torch.zeros(M, T_, d, device=S_paths.device)
    avg[:, 0, :] = S_paths[:, 0, :]
    if T_ > 1:
        cumsum = S_paths[:, 1:, :].cumsum(dim=1)
        counts = torch.arange(1, T_ + 1, device=S_paths.device,
                              dtype=torch.float32).unsqueeze(0).unsqueeze(2)
        avg[:, 1:, :] = (cumsum / counts)[:, :-1, :]
    return avg


def payoff_basket_asian_call(S_paths, kappa=1.0):
    R_bar = S_paths[:, 1:, :].mean(dim=1)
    return torch.clamp(R_bar.mean(dim=1) - kappa, min=0.0)


def payoff_asian_worst_of_put(S_paths, kappa=1.0):
    R_bar = S_paths[:, 1:, :].mean(dim=1)
    return torch.clamp(kappa - R_bar.min(dim=1).values, min=0.0)


PAYOFF_FNS = {
    'basket_asian_call':   payoff_basket_asian_call,
    'asian_worst_of_put':  payoff_asian_worst_of_put,
}


class HedgingNetwork(nn.Module):
    """Feedforward hedging network — must match training architecture exactly."""
    def __init__(self, d=3, hidden=(64, 64)):
        super().__init__()
        self.d = d
        self.hidden = tuple(hidden)
        layers = []
        prev = 3 * d + 1
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, d))
        self.net = nn.Sequential(*layers)
        nn.init.xavier_uniform_(self.net[-1].weight, gain=0.1)
        nn.init.zeros_(self.net[-1].bias)
        self.V0 = nn.Parameter(torch.tensor(0.0))

    def forward(self, spots, running_avg, delta_prev, time_left):
        x = torch.cat([spots, running_avg, delta_prev, time_left], dim=1)
        return self.net(x)

    @property
    def n_parameters(self):
        return sum(p.numel() for p in self.parameters())


def deep_hedge_forward(net, S_paths, payoff_fn, kappa=1.0, cost_rate=COST_RATE):
    """Forward pass through the hedging policy."""
    M, T1, d = S_paths.shape
    T_ = T1 - 1
    running_avg = compute_running_averages(S_paths)
    time_fracs = torch.arange(T_, 0, -1, device=S_paths.device,
                              dtype=torch.float32) / T_
    delta = torch.zeros(M, d, device=S_paths.device)
    pnl   = torch.zeros(M,    device=S_paths.device)
    cost  = torch.zeros(M,    device=S_paths.device)
    for t in range(T_):
        delta_new = net(S_paths[:, t, :], running_avg[:, t, :],
                        delta, time_fracs[t].expand(M, 1))
        cost = cost + cost_rate * ((delta_new - delta).abs()
                                   * S_paths[:, t, :]).sum(dim=1)
        pnl  = pnl  + (delta_new * (S_paths[:, t+1, :]
                                    - S_paths[:, t, :])).sum(dim=1)
        delta = delta_new
    payoff = payoff_fn(S_paths, kappa)
    return {
        'residuals': payoff - net.V0 - pnl + cost,
        'payoff':    payoff.detach(),
        'pnl':       pnl.detach(),
        'cost':      cost.detach(),
        'V0':        net.V0.item(),
    }


@torch.no_grad()
def evaluate_full(net, S_paths, payoff_fn, kappa=1.0, cost_rate=COST_RATE):
    """Full evaluation: residual stats including upper-tail-mean CVaR."""
    net.eval()
    r = deep_hedge_forward(net, S_paths, payoff_fn, kappa, cost_rate)
    res = r['residuals']
    n = len(res)
    s = torch.sort(res).values
    return {
        'rmse':     (res ** 2).mean().sqrt().item(),
        'mean':     res.mean().item(),
        'std':      res.std().item(),
        'cvar95':   s[int(np.ceil(0.95 * n)):].mean().item(),
        'cvar99':   s[int(np.ceil(0.99 * n)):].mean().item(),
        'max':      res.max().item(),
        'min':      res.min().item(),
        'V0':       r['V0'],
        'avg_cost': r['cost'].mean().item(),
        'avg_pnl':  r['pnl'].mean().item(),
        'n_paths':  int(n),
    }


print(f"✓ Model components ready ({HedgingNetwork(d=3).n_parameters} params)")


Device: cuda
✓ Model components ready (5060 params)


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Re-evaluate ALL 360 checkpoints on ALL 3 OOS periods
#
# Khác phiên bản trước: KHÔNG reuse training-time JSONs.
# Tất cả 3 periods được eval đồng nhất với cùng code path.
# Total = 360 ckpt × 3 periods = 1,080 evaluations.
# Runtime ≈ 4-5 phút trên T4.
#
# Outlier handling — SBTS / asian_worst_of_put / κ=0.95 / seed=3:
#   Cấu hình này phân kỳ trong CVaR phase (std_COVID ~ 1.7e10).
#   Dùng checkpoint của seed=10 (rerun convergent, từ Notebook 3 Cell 6)
#   nhưng LƯU vào key seed=3 để paired t-test giữ n=10.
# ═══════════════════════════════════════════════════════════════════
EVAL_FILE = DRIVE_FOLDER / f'evaluation_all_periods_{RUN_TAG}.json'

OUTLIER_REPLACEMENT = {
    ('SBTS', 'asian_worst_of_put', 0.95, 3): 10,
}

if EVAL_FILE.exists():
    print(f"✓ Cached: {EVAL_FILE.name}")
    with open(EVAL_FILE) as f:
        eval_results = json.load(f)
    n_entries = len(eval_results)
    n_phase_period = sum(
        len(v.get(phase, {})) for v in eval_results.values() for phase in PHASES
    )
    print(f"  {n_entries} keys, {n_phase_period} phase-period entries")
else:
    cached = np.load(HIST_PATHS_FILE)
    period_tensors = {
        p: torch.tensor(cached[f'S_norm_{p}'], dtype=torch.float32, device=DEVICE)
        for p in PERIODS
    }
    print("Period tensor shapes:")
    for p, t in period_tensors.items():
        print(f"  {p}: {tuple(t.shape)}")

    eval_results = {}
    total = (len(DS_NAMES) * len(OPTION_NAMES) * len(KAPPA_LEVELS)
             * N_SEEDS * len(PHASES))
    counter, n_ok, n_fail, n_swap = 0, 0, 0, 0
    t_start = time.time()

    print(f"\nRe-evaluating {total} (ckpt, phase) combinations on 3 periods each...\n")

    for ds, opt, kappa, seed, phase in product(
            DS_NAMES, OPTION_NAMES, KAPPA_LEVELS, range(N_SEEDS), PHASES):
        counter += 1
        actual_seed = seed
        swap_key = (ds, opt, kappa, seed)
        if swap_key in OUTLIER_REPLACEMENT:
            actual_seed = OUTLIER_REPLACEMENT[swap_key]
            n_swap += 1

        fname = f'{ds}_article_{opt}_k{kappa:.2f}_s{actual_seed}_{phase}.pt'
        ckpt_path = CKPT_ROOT / ds / fname
        if not ckpt_path.exists():
            print(f"  ❌ Missing: {fname}")
            n_fail += 1
            continue

        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            net = HedgingNetwork(d=N_ASSETS).to(DEVICE)
            state = None
            for k in ['net_state_dict', 'model_state', 'state_dict', 'net_state']:
                if k in ckpt:
                    state = ckpt[k]
                    break
            if state is None:
                state = ckpt
            net.load_state_dict(state)
            net.eval()

            # Eval on ALL 3 periods
            key = f"{ds}__{opt}__k{kappa:.2f}__s{seed}"
            if key not in eval_results:
                eval_results[key] = {}
            eval_results[key][phase] = {}
            for period in PERIODS:
                m = evaluate_full(net, period_tensors[period],
                                  PAYOFF_FNS[opt], kappa)
                eval_results[key][phase][period] = m
            n_ok += 1

            if counter % 60 == 0:
                elapsed = time.time() - t_start
                eta = elapsed / counter * (total - counter)
                print(f"  [{counter:>4}/{total}]  ok={n_ok:>4}  fail={n_fail:>2}  "
                      f"swap={n_swap:>2}  elapsed={elapsed:>5.0f}s  "
                      f"eta={eta:>5.0f}s")
        except Exception as e:
            print(f"  ❌ {fname}: {e}")
            n_fail += 1

    with open(EVAL_FILE, 'w') as f:
        json.dump(eval_results, f, indent=2)

    print(f"\n✓ Saved: {EVAL_FILE}")
    print(f"  ok = {n_ok} / {total}, fail = {n_fail}, swap = {n_swap}")
    print(f"  Total time: {(time.time()-t_start)/60:.1f} min")


Period tensor shapes:
  COVID_2019_2020: (505, 253, 3)
  PostCOVID_2021_22: (503, 253, 3)
  Recent_2023_25: (499, 253, 3)

Re-evaluating 360 (ckpt, phase) combinations on 3 periods each...

  [  60/360]  ok=  60  fail= 0  swap= 0  elapsed=   41s  eta=  203s
  [ 120/360]  ok= 120  fail= 0  swap= 0  elapsed=   75s  eta=  150s
  [ 180/360]  ok= 180  fail= 0  swap= 0  elapsed=  110s  eta=  110s
  [ 240/360]  ok= 240  fail= 0  swap= 0  elapsed=  145s  eta=   73s
  [ 300/360]  ok= 300  fail= 0  swap= 0  elapsed=  179s  eta=   36s
  [ 360/360]  ok= 360  fail= 0  swap= 2  elapsed=  211s  eta=    0s

✓ Saved: /content/drive/MyDrive/ARTICLE_SBTS/evaluation_all_periods_3p_covid_ext.json
  ok = 360 / 360, fail = 0, swap = 2
  Total time: 3.5 min


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Build master DataFrame (per-seed × phase × period)
# Source: EVAL_FILE only (re-evaluated all periods uniformly).
# Đã bake seed=3 → seed=10 swap ngay từ CELL 4, không cần splice nữa.
# ═══════════════════════════════════════════════════════════════════
with open(EVAL_FILE) as f:
    eval_results = json.load(f)


def parse_key(key):
    parts = key.split('__')
    return (parts[0], parts[1],
            float(parts[2].replace('k', '')),
            int(parts[3].replace('s', '')))


rows = []
for key, payload in eval_results.items():
    ds, opt, kappa, seed = parse_key(key)
    for phase, periods_dict in payload.items():
        for period, pm in periods_dict.items():
            if period not in PERIODS:
                continue
            rows.append({
                'ds': ds, 'option': opt, 'kappa': kappa, 'seed': seed,
                'phase': phase, 'period': period,
                'std':    pm.get('std'),
                'cvar95': pm.get('cvar95'),
                'cvar99': pm.get('cvar99'),
                'rmse':   pm.get('rmse'),
                'max':    pm.get('max'),
                'mean':   pm.get('mean'),
                'V0':     pm.get('V0'),
            })

df = pd.DataFrame(rows)

print(f"Master DataFrame: {df.shape}")
print(f"\nBy phase:\n{df['phase'].value_counts()}")
print(f"\nBy period:\n{df['period'].value_counts()}")

# Sanity: each (ds, opt, k, phase, period) must have exactly N_SEEDS rows
expected = N_SEEDS
counts = df.groupby(['ds', 'option', 'kappa', 'phase', 'period']).size()
miss = counts[counts != expected]
if len(miss) > 0:
    print(f"\n⚠ {len(miss)} cells with seed count ≠ {expected}:")
    print(miss.head(10))
else:
    print(f"\n✓ All {len(counts)} cells have exactly {expected} seeds")

df.to_csv(RESULTS_DIR / f'per_seed_metrics_{RUN_TAG}.csv', index=False)
print(f"\n💾 {RESULTS_DIR / f'per_seed_metrics_{RUN_TAG}.csv'}")


Master DataFrame: (1080, 13)

By phase:
phase
mse     540
cvar    540
Name: count, dtype: int64

By period:
period
COVID_2019_2020      360
PostCOVID_2021_22    360
Recent_2023_25       360
Name: count, dtype: int64

✓ All 108 cells have exactly 10 seeds

💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/per_seed_metrics_3p_covid_ext.csv


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Sanity check seed=3 outlier swap
# Verify that SBTS / asian_worst_of_put / κ=0.95 / seed=3 entry
# now contains seed=10's metrics (network converged correctly).
# ═══════════════════════════════════════════════════════════════════
print("Outlier swap verification — SBTS / asian_worst_of_put / κ=0.95 / seed=3")
print("=" * 75)

target = df[(df['ds'] == 'SBTS')
            & (df['option'] == 'asian_worst_of_put')
            & (df['kappa'] == 0.95)
            & (df['seed'] == 3)]

if len(target) == 0:
    print("⚠ No rows for seed=3 in this cell")
else:
    print("\nseed=3 (now containing seed=10 numerics):")
    print(target[['phase', 'period', 'std', 'cvar95', 'V0']].round(4).to_string(index=False))

    siblings = df[(df['ds'] == 'SBTS')
                  & (df['option'] == 'asian_worst_of_put')
                  & (df['kappa'] == 0.95)
                  & (df['seed'] != 3)
                  & (df['phase'] == 'cvar')]

    print("\nSibling seeds (seeds 0-2,4-9, cvar phase):")
    print(siblings.groupby('period')['std'].agg(['mean', 'std', 'min', 'max']).round(4))

    print("\nSeed=3 std vs sibling range — all 3 periods, cvar phase:")
    for period in PERIODS:
        s3 = target[(target['phase'] == 'cvar')
                    & (target['period'] == period)]['std'].values
        sib_vals = siblings[siblings['period'] == period]['std']
        if len(s3) and len(sib_vals):
            in_range = (s3[0] >= sib_vals.min() * 0.5) and (s3[0] <= sib_vals.max() * 5)
            ok = "✓" if in_range else "⚠"
            print(f"  {ok} {period:<22s} seed=3: {s3[0]:.4f}  "
                  f"siblings: [{sib_vals.min():.4f}, {sib_vals.max():.4f}]")


Outlier swap verification — SBTS / asian_worst_of_put / κ=0.95 / seed=3

seed=3 (now containing seed=10 numerics):
phase            period    std  cvar95     V0
  mse   COVID_2019_2020 0.0325  0.0656 0.0741
  mse PostCOVID_2021_22 0.0188  0.0445 0.0741
  mse    Recent_2023_25 0.0091  0.0196 0.0741
 cvar   COVID_2019_2020 0.0484  0.0511 0.0741
 cvar PostCOVID_2021_22 0.0157  0.0329 0.0741
 cvar    Recent_2023_25 0.0103  0.0070 0.0741

Sibling seeds (seeds 0-2,4-9, cvar phase):
                     mean     std     min     max
period                                           
COVID_2019_2020    0.0580  0.0221  0.0387  0.1143
PostCOVID_2021_22  0.0189  0.0041  0.0132  0.0273
Recent_2023_25     0.0118  0.0013  0.0097  0.0139

Seed=3 std vs sibling range — all 3 periods, cvar phase:
  ✓ COVID_2019_2020        seed=3: 0.0484  siblings: [0.0387, 0.1143]
  ✓ PostCOVID_2021_22      seed=3: 0.0157  siblings: [0.0132, 0.0273]
  ✓ Recent_2023_25         seed=3: 0.0103  siblings: [0.0097, 0.0139]


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Paired t-tests + Cohen's d + Benjamini–Hochberg FDR
#
# Test family:
#   3 pair-comparisons × 2 options × 3 strikes × 3 periods × 2 phases
#                     × 2 metrics (std, cvar95) = 216 paired t-tests
# ═══════════════════════════════════════════════════════════════════
COMPARISONS = [
    ('SBTS',   'GBM',    'SBTS vs GBM'),
    ('SBTS',   'Heston', 'SBTS vs Heston'),
    ('Heston', 'GBM',    'Heston vs GBM'),
]


def cohens_d(x, y):
    diff = x - y
    sd = diff.std(ddof=1)
    return diff.mean() / sd if sd > 1e-10 else np.nan


def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


stat_rows = []
for opt in OPTION_NAMES:
    for kappa in KAPPA_LEVELS:
        for phase in PHASES:
            for period in PERIODS:
                for ds_a, ds_b, label in COMPARISONS:
                    for metric in ['std', 'cvar95']:
                        sub_a = df[(df['ds'] == ds_a) & (df['option'] == opt)
                                   & (df['kappa'] == kappa) & (df['phase'] == phase)
                                   & (df['period'] == period)]
                        sub_b = df[(df['ds'] == ds_b) & (df['option'] == opt)
                                   & (df['kappa'] == kappa) & (df['phase'] == phase)
                                   & (df['period'] == period)]
                        common_seeds = set(sub_a['seed']) & set(sub_b['seed'])
                        if len(common_seeds) < 2:
                            continue
                        a = sub_a[sub_a['seed'].isin(common_seeds)
                                  ].sort_values('seed')[metric].values
                        b = sub_b[sub_b['seed'].isin(common_seeds)
                                  ].sort_values('seed')[metric].values
                        t, p = sp_stats.ttest_rel(a, b)
                        stat_rows.append({
                            'comparison': label,
                            'option': opt, 'kappa': kappa,
                            'phase': phase, 'period': period, 'metric': metric,
                            'mean_a': float(a.mean()), 'mean_b': float(b.mean()),
                            'mean_diff': float(a.mean() - b.mean()),
                            't_stat': float(t), 'p_value': float(p),
                            'cohens_d': cohens_d(a, b),
                            'n_seeds': len(common_seeds),
                        })

df_tests = pd.DataFrame(stat_rows)

# ── Benjamini–Hochberg FDR ───────────────────────────────────────
m = len(df_tests)
df_tests = df_tests.sort_values('p_value').reset_index(drop=True)
df_tests['rank'] = np.arange(1, m + 1)
df_tests['p_bh'] = (df_tests['p_value'] * m / df_tests['rank']).clip(upper=1.0)
df_tests['p_bh'] = df_tests['p_bh'][::-1].cummin()[::-1]
df_tests['stars'] = df_tests['p_bh'].apply(sig_stars)
df_tests['sig_05'] = df_tests['p_bh'] < 0.05

print(f"Total tests: {m}")
print(f"Significant (p_BH < 0.05): {df_tests['sig_05'].sum()}")
print(f"\n=== Significant tests on σ(R), CVaR phase, by period ===")
sub = df_tests[(df_tests['metric'] == 'std') & (df_tests['phase'] == 'cvar')
               & df_tests['sig_05']]
for period in PERIODS:
    n = (sub['period'] == period).sum()
    print(f"  {PERIOD_LABELS[period]:<25s}: {n}")

print(f"\n=== Pairwise win counts (σ(R), CVaR phase, p_BH < 0.05) ===")
for ds_a, ds_b, label in COMPARISONS:
    sub_pair = df_tests[(df_tests['comparison'] == label)
                        & (df_tests['metric'] == 'std')
                        & (df_tests['phase'] == 'cvar')
                        & df_tests['sig_05']]
    a_wins = (sub_pair['mean_a'] < sub_pair['mean_b']).sum()
    b_wins = (sub_pair['mean_a'] > sub_pair['mean_b']).sum()
    print(f"  {label:<22s}: {ds_a} wins {a_wins}  |  {ds_b} wins {b_wins}")

df_tests.to_csv(RESULTS_DIR / f'statistical_tests_{RUN_TAG}.csv', index=False)
print(f"\n✓ Saved: {RESULTS_DIR / f'statistical_tests_{RUN_TAG}.csv'}")


Total tests: 216
Significant (p_BH < 0.05): 133

=== Significant tests on σ(R), CVaR phase, by period ===
  COVID 2019--2020         : 10
  PostCOVID 2021--2022     : 11
  Recent 2023--2025        : 14

=== Pairwise win counts (σ(R), CVaR phase, p_BH < 0.05) ===
  SBTS vs GBM           : SBTS wins 9  |  GBM wins 7
  SBTS vs Heston        : SBTS wins 8  |  Heston wins 6
  Heston vs GBM         : Heston wins 3  |  GBM wins 2

✓ Saved: /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/statistical_tests_3p_covid_ext.csv


In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Hansen–Lunde–Nason (2011) Model Confidence Set
# 18 cells (3 periods × 2 options × 3 strikes), 5,000 bootstrap.
# Loss = σ(R), CVaR phase, α = 0.10.
# ═══════════════════════════════════════════════════════════════════
def mcs_seed_level(losses, alpha=0.10, n_bootstrap=5000, seed=42):
    """HLN-2011 MCS on seed-level losses (T_max statistic)."""
    rng = np.random.default_rng(seed)
    models = list(losses.keys())
    n_seeds = len(next(iter(losses.values())))
    for m in models:
        if len(losses[m]) != n_seeds:
            raise ValueError(f"Unequal seeds for {m}")
    L = np.column_stack([losses[m] for m in models])
    remaining = list(range(len(models)))
    elimination = []
    pvals = {}

    while len(remaining) > 1:
        sub = L[:, remaining]
        d_bar = sub.mean(axis=0)
        var_d = sub.var(axis=0, ddof=1) / n_seeds
        std_d = np.sqrt(np.maximum(var_d, 1e-12))
        t_i = (d_bar - d_bar.mean()) / std_d
        t_max = float(t_i.max())

        boot = np.empty(n_bootstrap)
        for b in range(n_bootstrap):
            idx = rng.integers(0, n_seeds, size=n_seeds)
            sb = sub[idx]
            db_b = sb.mean(axis=0)
            vd_b = sb.var(axis=0, ddof=1) / n_seeds
            sd_b = np.sqrt(np.maximum(vd_b, 1e-12))
            rec = (db_b - d_bar) - (db_b - d_bar).mean()
            boot[b] = (rec / sd_b).max()
        p = float(np.mean(boot >= t_max))
        names = tuple(models[i] for i in remaining)
        pvals[names] = p
        if p > alpha:
            break
        worst = remaining[int(np.argmax(t_i))]
        elimination.append((models[worst], p))
        remaining.remove(worst)

    return [models[i] for i in remaining], {'eliminations': elimination, 'pvalues': pvals}


# ── Sanity check ────────────────────────────────────────────────
rng = np.random.default_rng(0)
demo_clear = {'A': 0.01 + 0.001 * rng.standard_normal(10),
              'B': 0.05 + 0.001 * rng.standard_normal(10),
              'C': 0.05 + 0.001 * rng.standard_normal(10)}
mcs_clear, _ = mcs_seed_level(demo_clear, n_bootstrap=2000)
demo_tie = {'A': 0.05 + 0.005 * rng.standard_normal(10),
            'B': 0.05 + 0.005 * rng.standard_normal(10),
            'C': 0.05 + 0.005 * rng.standard_normal(10)}
mcs_tie, _ = mcs_seed_level(demo_tie, n_bootstrap=2000)
print(f"Sanity: clear winner → {mcs_clear} (expect ['A'])")
print(f"Sanity: no winner    → {sorted(mcs_tie)} (expect ['A','B','C'])")
assert mcs_clear == ['A'] and sorted(mcs_tie) == ['A', 'B', 'C']

# ── Run MCS on all 18 cells ─────────────────────────────────────
LOSS_COL  = 'std'
PHASE_RUN = 'cvar'
ALPHA     = 0.10
N_BOOT    = 5000

sub_df = df[df['phase'] == PHASE_RUN].copy()
mcs_rows = []
print(f"\nRunning MCS on {len(PERIODS) * len(OPTION_NAMES) * len(KAPPA_LEVELS)} cells "
      f"({N_BOOT} bootstrap each)...")
t_mcs = time.time()

for period in PERIODS:
    for option in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            cell = sub_df[(sub_df['period'] == period)
                          & (sub_df['option'] == option)
                          & (sub_df['kappa'] == kappa)]
            losses = {}
            for ds in DS_NAMES:
                vals = cell[cell['ds'] == ds].sort_values('seed')[LOSS_COL].values
                if len(vals) > 0:
                    losses[ds] = vals.astype(float)
            if len(losses) < 2:
                continue
            sizes = {len(v) for v in losses.values()}
            if len(sizes) > 1:
                print(f"  Skip {period}/{option}/k={kappa}: unequal seeds")
                continue
            mcs, det = mcs_seed_level(losses, alpha=ALPHA,
                                      n_bootstrap=N_BOOT, seed=42)
            means = {m: float(np.mean(v)) for m, v in losses.items()}
            best = min(means, key=means.get)
            mcs_rows.append({
                'period': period, 'option': option, 'kappa': kappa,
                'mcs': ','.join(sorted(mcs)),
                'mcs_size': len(mcs),
                'best_mean': best,
                'gbm_mean':    means.get('GBM'),
                'heston_mean': means.get('Heston'),
                'sbts_mean':   means.get('SBTS'),
            })

mcs_df = pd.DataFrame(mcs_rows)
mcs_df.to_csv(RESULTS_DIR / f'mcs_{RUN_TAG}.csv', index=False)
print(f"  Done in {time.time() - t_mcs:.1f}s")
print(f"  💾 {RESULTS_DIR / f'mcs_{RUN_TAG}.csv'}  ({len(mcs_df)} cells)")

# ── Summary ─────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("MCS membership (★ = singleton — statistically preferred generator)")
print("=" * 72)
for period in PERIODS:
    sub = mcs_df[mcs_df['period'] == period]
    print(f"\n{PERIOD_LABELS[period]}:")
    for _, r in sub.iterrows():
        star = '★' if r['mcs_size'] == 1 else ' '
        print(f"  {star} {r['option']:<22s} κ={r['kappa']:.2f}: "
              f"{{{r['mcs']}}}  (best-mean: {r['best_mean']})")

print("\n" + "─" * 72)
print(f"Singleton MCS counts by period:")
for period in PERIODS:
    sub = mcs_df[mcs_df['period'] == period]
    n_single = (sub['mcs_size'] == 1).sum()
    n_total = len(sub)
    print(f"  {PERIOD_LABELS[period]:<25s}: {n_single}/{n_total} singleton")

# Overall winner counts
print(f"\nWho appears in MCS (across all 18 cells)?")
for ds in DS_NAMES:
    n = mcs_df['mcs'].str.contains(ds).sum()
    print(f"  {ds}: {n}/{len(mcs_df)} cells "
          f"(singleton: {((mcs_df['mcs'] == ds)).sum()})")


Sanity: clear winner → ['A'] (expect ['A'])
Sanity: no winner    → ['A', 'B', 'C'] (expect ['A','B','C'])

Running MCS on 18 cells (5000 bootstrap each)...
  Done in 10.9s
  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/mcs_3p_covid_ext.csv  (18 cells)

MCS membership (★ = singleton — statistically preferred generator)

COVID 2019--2020:
    basket_asian_call      κ=0.95: {GBM,Heston,SBTS}  (best-mean: Heston)
    basket_asian_call      κ=1.00: {GBM,Heston,SBTS}  (best-mean: Heston)
    basket_asian_call      κ=1.05: {GBM,Heston,SBTS}  (best-mean: GBM)
    asian_worst_of_put     κ=0.95: {GBM,Heston,SBTS}  (best-mean: Heston)
  ★ asian_worst_of_put     κ=1.00: {Heston}  (best-mean: Heston)
    asian_worst_of_put     κ=1.05: {GBM,Heston,SBTS}  (best-mean: GBM)

PostCOVID 2021--2022:
  ★ basket_asian_call      κ=0.95: {SBTS}  (best-mean: SBTS)
    basket_asian_call      κ=1.00: {GBM,Heston,SBTS}  (best-mean: SBTS)
  ★ basket_asian_call      κ=1.05: {SBTS}  (best-mean:

In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — LaTeX tables (no stress tables)
#
# Tables generated:
#   - tab_main_results.tex      σ(R) on Recent_2023_25 + p_BH
#   - tab_scoreboard.tex        3-way pairwise win counts
#   - tab_scoreboard_detail.tex Cell-by-cell SBTS matchups
#   - tab_mcs.tex               MCS membership across 18 cells
#   - tab_v0.tex                Learned price offset V0
#   - tab_curriculum.tex        MSE-only vs curriculum (κ=1.00)
# ═══════════════════════════════════════════════════════════════════
def write_tex(path, content):
    with open(path, 'w') as f:
        f.write(content)
    print(f'  💾 {path}')


# ── Table 1: Main results ───────────────────────────────────────
def table_main_results():
    lines = [
        r'\begin{table}[H]', r'\centering', r'\small',
        r'\caption{Hedging-error standard deviation $\sigma(R)$ in '
        r'Recent 2023--2025 (Phase~2, $n = 10$). Improvement is relative '
        r'to \GBM. \textbf{Bold} indicates best per row.}',
        r'\label{tab:main-results}',
        r'\begin{tabular}{@{}llcccrr@{}}', r'\toprule',
        r'Option & $\kappa$ & \GBM & \Heston & \SBTS &'
        r' \textbf{$\Delta$ vs \GBM} & $p_{\mathrm{BH}}$ \\',
        r'\midrule',
    ]
    sub = df[(df['phase'] == 'cvar') & (df['period'] == BASELINE_PERIOD)]
    for opt in OPTION_NAMES:
        for i, kappa in enumerate(KAPPA_LEVELS):
            cell = sub[(sub['option'] == opt) & (sub['kappa'] == kappa)]
            stats = {ds: (cell[cell['ds'] == ds]['std'].mean(),
                          cell[cell['ds'] == ds]['std'].std())
                     for ds in DS_NAMES}
            best = min(stats, key=lambda d: stats[d][0])
            cells_str = []
            for ds in DS_NAMES:
                m, s = stats[ds]
                txt = f'${m:.4f}\\pm{s:.4f}$'
                if ds == best:
                    txt = r'$\mathbf{' + f'{m:.4f}\\pm{s:.4f}' + '}$'
                cells_str.append(txt)
            improv = (stats['GBM'][0] - stats['SBTS'][0]) / stats['GBM'][0] * 100
            t = df_tests[(df_tests['comparison'] == 'SBTS vs GBM')
                         & (df_tests['option'] == opt)
                         & (df_tests['kappa'] == kappa)
                         & (df_tests['phase'] == 'cvar')
                         & (df_tests['period'] == BASELINE_PERIOD)
                         & (df_tests['metric'] == 'std')]
            if len(t) == 0:
                pbh_str = '---'
            else:
                pbh = float(t.iloc[0]['p_bh']); stars = t.iloc[0]['stars']
                pbh_str = r'$<0.001$' if pbh < 0.001 else f'${pbh:.3f}$'
                if stars != 'ns':
                    pbh_str += f',{stars}'
            opt_label = OPTION_LABELS[opt] if i == 0 else ''
            multirow = (r'\multirow{3}{*}{' + opt_label + '}') if i == 0 else ''
            lines.append(f'  {multirow} & {kappa:.2f} & ' + ' & '.join(cells_str)
                         + f' & ${improv:+.0f}\\%$ & {pbh_str} \\\\')
        if opt != OPTION_NAMES[-1]:
            lines.append(r'\midrule')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


write_tex(RESULTS_DIR / 'tab_main_results.tex', table_main_results())


# ── Table 2: Scoreboard ─────────────────────────────────────────
def table_scoreboard():
    sub = df_tests[(df_tests['metric'] == 'std') & (df_tests['phase'] == 'cvar')
                   & df_tests['period'].isin(PERIODS)]
    rows = []
    for ds_a, ds_b, label in COMPARISONS:
        m = sub[sub['comparison'] == label]
        first  = ((m['mean_a'] < m['mean_b']) & m['sig_05']).sum()
        second = ((m['mean_a'] > m['mean_b']) & m['sig_05']).sum()
        ties   = (~m['sig_05']).sum()
        rows.append((label, first, second, ties))

    lines = [r'\begin{table}[H]', r'\centering', r'\small',
             r'\caption{Three-way pairwise comparison on $\sigma(R)$, '
             r'Phase~2, three OOS periods (COVID 2019--2020, PostCOVID '
             r'2021--2022, Recent 2023--2025). ``Wins'' are cells where '
             r'the first generator has a statistically lower mean loss '
             r'($p_{\mathrm{BH}}<0.05$); ``ties'' are cells in which '
             r'equality cannot be rejected after correction.}',
             r'\label{tab:scoreboard}',
             r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
             r'\textbf{Matchup} & \textbf{First wins} & '
             r'\textbf{Second wins} & \textbf{Ties} \\',
             r'\midrule']
    for label, w1, w2, t in rows:
        label_tex = (label.replace('SBTS', r'\SBTS')
                          .replace('GBM',  r'\GBM')
                          .replace('Heston', r'\Heston'))
        if w1 > w2 and w1 > t:
            w1_str = r'\textbf{' + str(w1) + '}'
        else:
            w1_str = str(w1)
        lines.append(f'  {label_tex} & {w1_str} & {w2} & {t} \\\\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


write_tex(RESULTS_DIR / 'tab_scoreboard.tex', table_scoreboard())


# ── Table 3: Scoreboard detail ──────────────────────────────────
def table_scoreboard_detail():
    lines = [r'\begin{table}[H]', r'\centering', r'\footnotesize',
             r'\setlength{\tabcolsep}{4pt}',
             r'\caption{Cell-by-cell outcomes in the two \SBTS\ '
             r'matchups across the $18$ comparisons (3 periods $\times$ '
             r'2 options $\times$ 3 strikes). Decided cells list the '
             r'winner with Benjamini--Hochberg-adjusted significance '
             r'(* $p_{\mathrm{BH}}<0.05$, ** $<0.01$, *** $<0.001$); '
             r'tied cells report the signed mean-loss difference '
             r'$\Delta = \bar{L}_A - \bar{L}_B$ and the BH-adjusted '
             r'$p$-value. Loss is Phase~2 $\sigma(R)$.}',
             r'\label{tab:scoreboard-detail}',
             r'\begin{tabular}{@{}lllll@{}}', r'\toprule',
             r'\textbf{Period} & \textbf{Option} & $\kappa$'
             r' & \textbf{\SBTS\ vs \GBM} & \textbf{\SBTS\ vs \Heston} \\',
             r'\midrule']
    for p_idx, period in enumerate(PERIODS):
        for o_idx, opt in enumerate(OPTION_NAMES):
            for k_idx, kappa in enumerate(KAPPA_LEVELS):
                period_label = PERIOD_LABELS[period] if (o_idx == 0 and k_idx == 0) else ''
                if k_idx == 0:
                    period_cell = (r'\multirow{6}{*}{' + period_label + '}'
                                   if o_idx == 0 else '')
                    opt_cell = r'\multirow{3}{*}{' + OPTION_LABELS[opt] + '}'
                else:
                    period_cell = ''; opt_cell = ''
                cells_out = []
                for ds_other, label in [('GBM', 'SBTS vs GBM'),
                                         ('Heston', 'SBTS vs Heston')]:
                    t = df_tests[(df_tests['comparison'] == label)
                                 & (df_tests['option'] == opt)
                                 & (df_tests['kappa'] == kappa)
                                 & (df_tests['phase'] == 'cvar')
                                 & (df_tests['period'] == period)
                                 & (df_tests['metric'] == 'std')]
                    if len(t) == 0:
                        cells_out.append('---'); continue
                    r = t.iloc[0]
                    if r['sig_05']:
                        winner = 'SBTS' if r['mean_a'] < r['mean_b'] else ds_other
                        winner_tex = (r'\SBTS' if winner == 'SBTS'
                                      else (r'\GBM' if winner == 'GBM' else r'\Heston'))
                        cells_out.append(winner_tex + r'\textsuperscript{' + r['stars'] + '}')
                    else:
                        delta = r['mean_a'] - r['mean_b']
                        sign = '{+}' if delta >= 0 else '{-}'
                        cells_out.append('tie ($\\Delta{=}' + sign
                                         + f'{abs(delta):.4f}$, $p{{=}}{r["p_bh"]:.2f}$)')
                lines.append(f'  {period_cell} & {opt_cell} & {kappa:.2f} & '
                             + ' & '.join(cells_out) + r' \\')
            if o_idx == 0:
                lines.append(r'\cmidrule(lr){2-5}')
        if p_idx < len(PERIODS) - 1:
            lines.append(r'\midrule')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


write_tex(RESULTS_DIR / 'tab_scoreboard_detail.tex', table_scoreboard_detail())


# ── Table 4: MCS ────────────────────────────────────────────────
def table_mcs():
    gen_fmt = {'GBM': r'\GBM', 'Heston': r'\Heston', 'SBTS': r'\SBTS'}
    lines = [r'\begin{table}[H]', r'\centering', r'\small',
             r'\caption{Model Confidence Set membership by regime, '
             r'option, and strike, computed via the~\citet{HansenLundeNason2011} '
             r'procedure at $\alpha = 0.10$ on the seed-level loss '
             r'statistic $L = \sigma(R)$ with $5{,}000$ bootstrap '
             r'resamples. Entries list generators whose predictive '
             r'ability cannot be distinguished from the best at the '
             r'10\% level. \textbf{Bold} marks singleton MCS (statistically '
             r'preferred generator).}',
             r'\label{tab:mcs}',
             r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
             r'\textbf{Period} & \textbf{Option} & $\kappa=0.95$ '
             r'& $\kappa=1.00$ & $\kappa=1.05$ \\',
             r'\midrule']
    for p_idx, period in enumerate(PERIODS):
        for o_idx, opt in enumerate(OPTION_NAMES):
            cells_str = []
            for kappa in KAPPA_LEVELS:
                row = mcs_df[(mcs_df['period'] == period)
                             & (mcs_df['option'] == opt)
                             & (mcs_df['kappa'] == kappa)]
                if len(row) == 0:
                    cells_str.append('---'); continue
                mcs_set = row.iloc[0]['mcs'].split(',')
                rendered = ', '.join(gen_fmt[m] for m in mcs_set)
                if len(mcs_set) == 1:
                    rendered = r'\textbf{' + rendered + '}'
                cells_str.append(rendered)
            period_label = (r'\multirow{2}{*}{' + PERIOD_LABELS[period] + '}'
                            if o_idx == 0 else '')
            lines.append(f'  {period_label} & {OPTION_LABELS[opt]} & '
                         + ' & '.join(cells_str) + r' \\')
        if p_idx < len(PERIODS) - 1:
            lines.append(r'\midrule')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


write_tex(RESULTS_DIR / 'tab_mcs.tex', table_mcs())


# ── Table 5: V0 ─────────────────────────────────────────────────
def table_v0():
    lines = [r'\begin{table}[H]', r'\centering', r'\small',
             r'\caption{Learned price offset $V_0^{*}$ (mean across 10 '
             r'seeds), normalised by $S_0 = 1$.}',
             r'\label{tab:v0}',
             r'\begin{tabular}{@{}lcccccc@{}}', r'\toprule',
             r' & \multicolumn{3}{c}{\textbf{Basket call}}'
             r' & \multicolumn{3}{c}{\textbf{Worst-of put}} \\',
             r'\cmidrule(lr){2-4}\cmidrule(lr){5-7}',
             r'\textbf{Generator} & $\kappa{=}0.95$ & $\kappa{=}1.00$ & '
             r'$\kappa{=}1.05$ & $\kappa{=}0.95$ & $\kappa{=}1.00$ & '
             r'$\kappa{=}1.05$ \\',
             r'\midrule']
    # V0 is constant across periods (network parameter); read from BASELINE_PERIOD cvar
    sub = df[(df['phase'] == 'cvar') & (df['period'] == BASELINE_PERIOD)]
    means = {}
    for ds in DS_NAMES:
        means[ds] = {}
        for opt in OPTION_NAMES:
            for kappa in KAPPA_LEVELS:
                v = sub[(sub['ds'] == ds) & (sub['option'] == opt)
                        & (sub['kappa'] == kappa)]['V0'].mean()
                means[ds][(opt, kappa)] = v
    best = {(opt, kappa): min(DS_NAMES, key=lambda ds: means[ds][(opt, kappa)])
            for opt in OPTION_NAMES for kappa in KAPPA_LEVELS}
    for ds in DS_NAMES:
        ds_tex = {'GBM': r'\GBM', 'Heston': r'\Heston', 'SBTS': r'\SBTS'}[ds]
        cells_str = [ds_tex]
        for opt in OPTION_NAMES:
            for kappa in KAPPA_LEVELS:
                v = means[ds][(opt, kappa)]
                txt = f'{v:.4f}'
                if best[(opt, kappa)] == ds:
                    txt = r'\textbf{' + txt + '}'
                cells_str.append(txt)
        lines.append('  ' + ' & '.join(cells_str) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


write_tex(RESULTS_DIR / 'tab_v0.tex', table_v0())


# ── Table 6: Curriculum ────────────────────────────────────────
def table_curriculum():
    lines = [r'\begin{table}[H]', r'\centering', r'\small',
             r'\caption{Curriculum effect: MSE-only versus Phase~1 $+$ '
             r'Phase~2, Recent 2023--2025, $\kappa = 1.00$.}',
             r'\label{tab:curriculum}',
             r'\begin{tabular}{@{}llccr@{}}', r'\toprule',
             r'\textbf{Option} & \textbf{Generator} & '
             r'\textbf{MSE only} & \textbf{Curriculum} & '
             r'\textbf{Improvement} \\',
             r'\midrule']
    sub = df[df['period'] == BASELINE_PERIOD]
    for o_idx, opt in enumerate(OPTION_NAMES):
        for j, ds in enumerate(DS_NAMES):
            mse = sub[(sub['ds'] == ds) & (sub['option'] == opt)
                      & (sub['kappa'] == 1.00) & (sub['phase'] == 'mse')]['std'].mean()
            cur = sub[(sub['ds'] == ds) & (sub['option'] == opt)
                      & (sub['kappa'] == 1.00) & (sub['phase'] == 'cvar')]['std'].mean()
            impr = (mse - cur) / mse * 100 if mse > 0 else 0
            opt_cell = (r'\multirow{3}{*}{' + OPTION_LABELS[opt] + '}'
                        if j == 0 else '')
            ds_tex = {'GBM': r'\GBM', 'Heston': r'\Heston', 'SBTS': r'\SBTS'}[ds]
            cur_str  = f'${cur:.4f}$'
            impr_str = f'${impr:+.1f}\\%$'
            if ds == 'SBTS' and opt == 'basket_asian_call':
                cur_str  = r'$\mathbf{' + f'{cur:.4f}' + '}$'
                impr_str = r'$\mathbf{' + f'{impr:+.1f}\\%' + '}$'
            lines.append(f'  {opt_cell} & {ds_tex} & ${mse:.4f}$ & '
                         f'{cur_str} & {impr_str} \\\\')
        if o_idx == 0:
            lines.append(r'\midrule')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


write_tex(RESULTS_DIR / 'tab_curriculum.tex', table_curriculum())

print(f'\n✓ All LaTeX tables saved to {RESULTS_DIR}')


  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/tab_main_results.tex
  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/tab_scoreboard.tex
  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/tab_scoreboard_detail.tex
  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/tab_mcs.tex
  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/tab_v0.tex
  💾 /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/tab_curriculum.tex

✓ All LaTeX tables saved to /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext


In [10]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — Figure: Performance across regimes (CVaR phase, κ=0.95)
# Bỏ Fig 4 (stress degradation) vì không còn stress metrics.
# ═══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(r'Hedging Error $\sigma(R)$ Across Market Regimes (CVaR curriculum, $\kappa=0.95$)',
             fontsize=12, fontweight='bold', y=1.02)
bw = 0.22

for ax_idx, opt in enumerate(OPTION_NAMES):
    ax = axes[ax_idx]
    x = np.arange(len(PERIODS))
    for j, ds in enumerate(DS_NAMES):
        means, sems = [], []
        for period in PERIODS:
            v = df[(df['ds'] == ds) & (df['option'] == opt)
                   & (df['kappa'] == 0.95) & (df['phase'] == 'cvar')
                   & (df['period'] == period)]['std'].values
            means.append(v.mean() if len(v) > 0 else 0)
            sems.append(v.std() / np.sqrt(len(v)) if len(v) > 1 else 0)
        ax.bar(x + j * bw, means, bw, yerr=sems, capsize=3,
               label=ds, color=DS_COLORS[ds], alpha=0.85,
               edgecolor='white', linewidth=0.5)
    ax.set_xticks(x + bw)
    ax.set_xticklabels([PERIOD_LABELS[p].replace('--', '–') for p in PERIODS],
                       fontsize=8)
    ax.set_ylabel(r'$\sigma(R)$')
    ax.set_title(OPTION_LABELS[opt])
    ax.legend(framealpha=0.9)
    # Highlight COVID stress period
    ax.axvspan(-0.3, 0.7, alpha=0.08, color='red', zorder=0)

plt.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(FIG_DIR / f'fig_performance_across_regimes.{ext}',
                bbox_inches='tight')
plt.close(fig)
print(f"  💾 fig_performance_across_regimes (pdf + png) → {FIG_DIR}")


  💾 fig_performance_across_regimes (pdf + png) → /content/drive/MyDrive/ARTICLE_SBTS/article_results_3p_covid_ext/figures


In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — Final summary + outputs inventory
# ═══════════════════════════════════════════════════════════════════
print("═" * 75)
print(f" FINAL SUMMARY — 3 OOS PERIODS ".center(75, '═'))
print(f" Run tag: {RUN_TAG} ".center(75, '═'))
print("═" * 75)

for period in PERIODS:
    print(f"\n{PERIOD_LABELS[period]}")
    print("-" * 75)
    sub = df[(df['phase'] == 'cvar') & (df['period'] == period)]
    for opt in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            cell = sub[(sub['option'] == opt) & (sub['kappa'] == kappa)]
            line = f"  {OPTION_LABELS[opt]:<15s} κ={kappa:.2f}: "
            for ds in DS_NAMES:
                m = cell[cell['ds'] == ds]['std'].mean()
                line += f"{ds}={m:.4f}  "
            mcs_row = mcs_df[(mcs_df['period'] == period)
                              & (mcs_df['option'] == opt)
                              & (mcs_df['kappa'] == kappa)]
            if len(mcs_row) > 0 and mcs_row.iloc[0]['mcs_size'] == 1:
                line += f"   ★ MCS = {{{mcs_row.iloc[0]['mcs']}}}"
            elif len(mcs_row) > 0:
                line += f"   MCS = {{{mcs_row.iloc[0]['mcs']}}}"
            print(line)

print("\n" + "═" * 75)
print(" OUTPUTS ".center(75, '═'))
print("═" * 75)
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        rel = f.relative_to(RESULTS_DIR)
        size_kb = f.stat().st_size / 1024
        print(f"  {rel}  ({size_kb:.1f} KB)")

print(f"\n✅ Pipeline complete. Cache files in {DRIVE_FOLDER}:")
for cache_name in [f'historical_test_paths_{RUN_TAG}.npz',
                   f'evaluation_all_periods_{RUN_TAG}.json']:
    fp = DRIVE_FOLDER / cache_name
    if fp.exists():
        print(f"  ✓ {cache_name}  ({fp.stat().st_size / 1024 / 1024:.1f} MB)")


═══════════════════════════════════════════════════════════════════════════
══════════════════════ FINAL SUMMARY — 3 OOS PERIODS ══════════════════════
══════════════════════════ Run tag: 3p_covid_ext ══════════════════════════
═══════════════════════════════════════════════════════════════════════════

COVID 2019--2020
---------------------------------------------------------------------------
  Basket call     κ=0.95: GBM=0.0175  Heston=0.0172  SBTS=0.0425     MCS = {GBM,Heston,SBTS}
  Basket call     κ=1.00: GBM=0.0178  Heston=0.0171  SBTS=0.0348     MCS = {GBM,Heston,SBTS}
  Basket call     κ=1.05: GBM=0.0173  Heston=0.0173  SBTS=0.0340     MCS = {GBM,Heston,SBTS}
  Worst-of put    κ=0.95: GBM=0.0356  Heston=0.0337  SBTS=0.0570     MCS = {GBM,Heston,SBTS}
  Worst-of put    κ=1.00: GBM=0.0358  Heston=0.0341  SBTS=0.0578     ★ MCS = {Heston}
  Worst-of put    κ=1.05: GBM=0.0339  Heston=0.0351  SBTS=0.0869     MCS = {GBM,Heston,SBTS}

PostCOVID 2021--2022
-----------------------------